# Generalized spectral clustering

Clusters all 445 feature titles together, without filtering by product type. 

Our aim is to find attributes form groups that are strongly linked inside and only weakly linked to the rest.

We start from the shared similarity matrix: `3_similarity.ipynb` now saves it to `data/similarity.npz`, and this notebook (like the agglomerative and DBSCAN ones) just loads it.

**Method** (normalized spectral clustering):
1. We build a graph from the similarity matrix: attributes are nodes, similarities are edge weights (each node keeps its `K_NEIGHBORS` most similar neighbours).
2. We compute the normalized Laplacian and its eigenvectors with the smallest eigenvalues.
3. Finally, we run k-means on those eigenvectors

In [24]:
import json
import time
from collections import Counter

import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import HTML, display
from sklearn.cluster import KMeans
from scipy.sparse.csgraph import connected_components
from sklearn.metrics import davies_bouldin_score, silhouette_score

#links kept per attribute
K_NEIGHBORS = 15

#how many coordinates to compute at most
N_EIGEN = 60
K_RANGE = range(5, 61, 5)

#to make random parts repeatable
SEED = 0

#loading the similarity matrix (showing how alike each pair of attributes is)
with np.load("data/similarity.npz") as data:
    S = data["similarity"]
    #list of the attributes in the same order as matched in the matrix
    attributes = data["titles"].tolist()


#preliminary checks on the similarity matrix
n = len(attributes)
assert S.shape == (n, n), f"matrix {S.shape} does not match {n} attributes"
assert len(set(attributes)) == n, "duplicate attribute names"
assert S.min() >= 0 and S.max() <= 1 + 1e-9, "values are not in the interval [0,1]"
assert np.allclose(S, S.T), "similarity must be symmetric"

print(f"Loaded {n} attributes and matrix {S.shape}")

Loaded 445 attributes and matrix (445, 445)


## Build the similarity graph

The diagonal (originally set as 1, since all attributes are obviously strongly similar (identical!) to themselves) is set to 0 (we want no self-loops to W and that a node's own entry doesn't take one of its k neighbour slots). 

With `K_NEIGHBORS`, each attribute keeps only its most similar neighbours; the result is symmetrised so that an edge is kept if either endpoint selects it. 

This makes the graph sparse, removes weak noisy links, and is what makes spectral clustering scale to larger catalogs.

In [25]:
def build_affinity(S, k_neighbors=None):
    W = S.copy()
    np.fill_diagonal(W, 0)

    #sorts each row from smallest to largest, take k-th value from the end of row, turn values into a column
    threshold = np.sort(W, axis=1)[:, -K_NEIGHBORS][:, None]

    #we check each pair (x,y): if x is in y´s top K OR viceversa, the edge is kept
    #otherwise it is set to 0 
    W = np.where((W >= threshold) | (W >= threshold.T), W, 0) 
    return W


W = build_affinity(S, K_NEIGHBORS)

#check W is symmetric
assert np.allclose(W, W.T)

#weighted degree of d_i: sum of the similarities on node i's edges
deg = W.sum(axis=1)
#how many neighbors each node has 
nbrs = (W > 0).sum(axis=1)
#number of connected components
n_comp, _ = connected_components(W > 0, directed=False)

print(f"Edges kept: {np.count_nonzero(np.triu(W))} of {n*(n-1)//2} ({np.count_nonzero(np.triu(W))/(n*(n-1)//2):.1%})")

print(f"Neighbours per node: min {nbrs.min()}, median {int(np.median(nbrs))}, max {nbrs.max()}")

print(f"Weighted degree: min {deg.min():.2f}, median {np.median(deg):.2f}, max {deg.max():.2f}")

print(f"Isolated nodes: {(deg == 0).sum()}, connected components: {n_comp}")

Edges kept: 4618 of 98790 (4.7%)
Neighbours per node: min 15, median 18, max 59
Weighted degree: min 1.01, median 5.60, max 18.85
Isolated nodes: 0, connected components: 1


## Normalized Laplacian and eigengap

`np.linalg.eigh` is used because $L_{sym}$ is symmetric; it returns eigenvalues in ascending order. Small eigenvalues correspond to well-separated groups; the largest jump (eigengap) among them suggests the number of clusters.

In [ ]:
def normalized_laplacian(W):
    # matrix whose eigenvectors give the best ways to split the graph into groups
    #L_sym = I - D^(-1/2) W D^(-1/2): each link w_ij is rescaled to w_ij / sqrt(d_i * d_j)
    
    #without it, very connected attributes would dominate and pull everything towards them
    #after rescaling, a link counts more when it’s important to both sides
    
    deg = W.sum(axis=1)   

    #computing 1/sqrt(d) for each attribute of deg                       
    d_inv_sqrt = 1 / np.sqrt(np.maximum(deg, 1e-12)) # 1e-12 avoids division by zero
    #eye build the idientity matrix, outer builds a multiplication table having d_inv_sqrt as both a row and a column
    #w is multiplied element wise
    return np.eye(len(W)) - W * np.outer(d_inv_sqrt, d_inv_sqrt)


def spectral_embedding(W, n_eigen):
    """Return the n_eigen smallest eigenvalues of L_sym and their eigenvectors."""
    eigenvalues, eigenvectors = np.linalg.eigh(normalized_laplacian(W))
    return eigenvalues[:n_eigen], eigenvectors[:, :n_eigen]

K_MIN, K_MAX = 20, 80

eigenvalues, eigenvectors = spectral_embedding(W, n_eigen=K_MAX + 1)

gaps = np.diff(eigenvalues)

# number of small eigenvalues before the biggest jump = number of clusters
k_eigengap = int(np.argmax(gaps[K_MIN - 1:K_MAX]) + K_MIN)

print(f"Eigengap between k={K_MIN} and {K_MAX} suggests k = {k_eigengap} "
      f"(gap = {gaps[k_eigengap - 1]:.4f})")

Eigengap between k=20 and 80 suggests k = 25


## 4. Spectral clustering

Rows of the first *k* eigenvectors are normalized to unit length (Ng–Jordan–Weiss step), then clustered with k-means.

In [5]:
def spectral_cluster(eigenvectors, k, seed=SEED):
    U = eigenvectors[:, :k]
    U = U / np.maximum(np.linalg.norm(U, axis=1, keepdims=True), 1e-12)
    labels = KMeans(n_clusters=k, n_init=10, random_state=seed).fit_predict(U)
    return labels, U


k = k_eigengap
labels, U = spectral_cluster(eigenvectors, k)
sizes = Counter(labels)
print(f"{k} clusters; sizes (largest first): {[size for _, size in sizes.most_common()]}")

25 clusters; sizes (largest first): [31, 30, 26, 24, 24, 21, 20, 20, 19, 19, 18, 18, 18, 18, 17, 17, 14, 14, 14, 13, 12, 11, 11, 9, 7]


## 5. Evaluation

TO BE COMPLETED

## 6. Cluster visualization

t-SNE projects the same precomputed distances (`1 - similarity`) to two dimensions for display only; it does not change the clustering labels or serve as an evaluation metric. Nearby points suggest similarity, but the projection can distort some distances. The largest groups get distinct colours, the others are grey, and singletons (if any) are hollow markers. Hover over a point to see its attribute and group.

In [6]:
from sklearn.manifold import TSNE

HIGHLIGHT_GROUPS = 8
HIGHLIGHT_PALETTE = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#4a3aa7", "#e34948"]
GRAY, LIGHT_GRAY, INK, MUTED = "#8c8c86", "#c9c8c0", "#1f2933", "#6b7785"

distance_matrix = 1 - similarity
multi = [label for label, size in sizes.items() if size > 1]

coordinates = TSNE(
    n_components=2,
    metric="precomputed",
    init="random",
    perplexity=30,
    random_state=42,
).fit_transform(distance_matrix)


def group_name(label):
    """Use the most centrally similar attribute as the group's display name."""
    indices = np.where(labels == label)[0]
    within_group_similarity = similarity[np.ix_(indices, indices)]
    medoid_index = indices[within_group_similarity.sum(axis=1).argmax()]
    return attributes[medoid_index]


ranked_groups = sorted(multi, key=lambda label: -sizes[label])
highlighted_groups = ranked_groups[:HIGHLIGHT_GROUPS]
group_names = {label: group_name(label) for label in multi}


def make_trace(indices, name, color, hollow=False, size=9):
    return go.Scatter(
        x=coordinates[indices, 0],
        y=coordinates[indices, 1],
        mode="markers",
        name=name,
        marker=dict(
            size=size,
            color="white" if hollow else color,
            line=dict(width=1.5 if hollow else 1, color=color if hollow else "white"),
        ),
        text=[attributes[index] for index in indices],
        customdata=[group_names.get(labels[index], "Singleton") for index in indices],
        hovertemplate="<b>%{text}</b><br>group: %{customdata}<extra></extra>",
    )


other_group_indices = np.array(
    [index for index, label in enumerate(labels) if sizes[label] > 1 and label not in highlighted_groups],
    dtype=int,
)
singleton_indices = np.array([index for index, label in enumerate(labels) if sizes[label] == 1], dtype=int)

figure = go.Figure()
figure.add_trace(
    make_trace(singleton_indices, f"Singletons ({len(singleton_indices)})", LIGHT_GRAY, hollow=True, size=7)
)
figure.add_trace(
    make_trace(other_group_indices, f"Other groups ({len(ranked_groups) - len(highlighted_groups)})", GRAY, size=7)
)
for color, label in zip(HIGHLIGHT_PALETTE, highlighted_groups):
    indices = np.where(labels == label)[0]
    figure.add_trace(make_trace(indices, f"{group_names[label]} ({sizes[label]})", color, size=10))

figure.update_layout(
    title=dict(text="Attribute clusters: spectral clustering (t-SNE)", x=0, font=dict(color=INK)),
    template="simple_white",
    width=1000,
    height=720,
    xaxis=dict(visible=False),
    yaxis=dict(visible=False),
    legend=dict(title="Largest groups", font=dict(color=INK), itemsizing="constant"),
    hoverlabel=dict(bgcolor="white", font_color=INK),
)
display(HTML(figure.to_html(full_html=False, include_plotlyjs="cdn")))